# 🎬 AnimeInterp Benchmark — ATD-12K Dataset

> **Model:** AnimeInterp (CVPR 2021) — *Deep Animation Video Interpolation in the Wild*  
> **Môi trường:** Kaggle Notebook · GPU T4 x2 hoặc P100  
> **Dataset:** ATD-12K `test_2k_540p` — Triplet format (frame1/frame2/frame3)  
> **Output:** 17 chỉ số chuẩn hóa → `benchmark_results_AnimeInterp.csv`

## 📋 Pipeline Tổng Quan

```
frame1.png (I0, t=0) ──►┐
                         ├─► AnimeInterp ──► frame_pred (t=0.5)
frame3.png (I1, t=1) ──►┘        │
                                   ▼
                    So sánh với frame2.png (GT)
                    → PSNR, SSIM, LPIPS, IE, Speed, VRAM
```

## 📊 Bảng 17 Chỉ Số Chuẩn

| # | Chỉ Số | Trạng Thái |
|---|--------|:-----------:|
| 1 | Chamfer Distance (CD) | ❌ N/A |
| 2 | Weighted CD (WCD) | ❌ N/A |
| 3 | **LPIPS** | ✅ Đo thực tế |
| 4 | EMD | ❌ N/A |
| 5 | RRLD | ❌ N/A |
| 6 | **PSNR** | ✅ Đo thực tế |
| 7 | **SSIM** | ✅ Đo thực tế |
| 8 | **IE** | ⚠️ Tùy chọn |
| 9 | FVD | ⚠️ N/A (single-frame) |
| 10 | Warping Error | ⚠️ N/A (single-frame) |
| 11 | Hausdorff Distance | ❌ N/A |
| 12 | Line IoU | ❌ N/A |
| 13 | Disconnection Rate | ❌ N/A |
| 14 | Arbitrary t Support | ❌ N/A (fixed t=0.5) |
| 15 | **Inference Speed** | ✅ Đo thực tế |
| 16 | **VRAM Peak** | ✅ Đo thực tế |
| 17 | Fine-tuning Speed | ⚠️ N/A |


---
## ⚙️ CELL 1: Thiết Lập Môi Trường & Cài Đặt Thư Viện

In [ ]:
# ============================================================
# CELL 1: Cài đặt thư viện và kiểm tra môi trường GPU
# ============================================================
import subprocess, sys

print('[STEP 1] Cài đặt thư viện cần thiết...')

packages_to_install = [
    'lpips',
    'opencv-python-headless',
    'scikit-image>=0.19',
    'scipy',
    'tqdm',
    'pandas',
    'matplotlib',
    'Pillow',
    'tabulate',
]

for pkg in packages_to_install:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'  {status} {pkg}')

print('\n[STEP 2] Kiểm tra môi trường...')
import torch
import torchvision

print(f'  Python         : {sys.version.split()[0]}')
print(f'  PyTorch        : {torch.__version__}')
print(f'  TorchVision    : {torchvision.__version__}')
print(f'  CUDA available : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU [{i}]         : {props.name} ({props.total_memory / 1024**3:.1f} GB VRAM)')
else:
    print('  ⚠️  Không phát hiện GPU! Vui lòng bật GPU trong Settings -> Accelerator.')

print('\n[STEP 3] nvidia-smi:')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                '--format=csv,noheader,nounits'], check=False)

---
## 📦 CELL 2: Chuẩn Bị Source Code & Load Model

In [ ]:
# ============================================================
# CELL 2A: Cấu hình đường dẫn (THAY ĐỔI THEO KAGGLE DATASET CỦA BẠN)
# ============================================================
import os

# Thay tên dataset Kaggle của bạn
ANIMEINTERP_ZIP      = '/kaggle/input/animeinterp-code/AnimeInterp_code.zip'
ANIMEINTERP_DIR      = '/kaggle/working/AnimeInterp'
ANIMEINTERP_DIR_DIRECT = '/kaggle/input/animeinterp-code/AnimeInterp'

# Dataset ATD-12K
DATASET_ROOT = '/kaggle/input/atd12k-benchmark/test_2k_540p'
FLOW_ROOT    = '/kaggle/input/atd12k-flows/test_2k_pre_calc_sgm_flows'

# Checkpoint
CHECKPOINT = '/kaggle/input/animeinterp-code/anime_interp_full.ckpt'

# Output
OUTPUT_DIR = '/kaggle/working/benchmark_AnimeInterp'
CSV_PATH   = os.path.join(OUTPUT_DIR, 'benchmark_results_AnimeInterp.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Cấu hình đường dẫn:')
print(f"  DATASET_ROOT : {DATASET_ROOT}  {'✅' if os.path.exists(DATASET_ROOT) else '❌ KHÔNG TỒN TẠI'}")
print(f"  FLOW_ROOT    : {FLOW_ROOT}  {'✅' if os.path.exists(FLOW_ROOT) else '⚠️ Không có -> dùng zero flow'}")
print(f"  CHECKPOINT   : {CHECKPOINT}  {'✅' if os.path.exists(CHECKPOINT) else '❌ KHÔNG TỒN TẠI'}")
print(f'  OUTPUT_DIR   : {OUTPUT_DIR}  ✅')

In [ ]:
# ============================================================
# CELL 2B: Giải nén code và thêm vào sys.path
# ============================================================
import sys, zipfile

code_dir = None

if os.path.isdir(ANIMEINTERP_DIR_DIRECT):
    code_dir = ANIMEINTERP_DIR_DIRECT
    print(f'[OK] Dùng trực tiếp: {code_dir}')

elif os.path.exists(ANIMEINTERP_ZIP):
    print(f'[INFO] Giải nén: {ANIMEINTERP_ZIP}')
    with zipfile.ZipFile(ANIMEINTERP_ZIP, 'r') as z:
        z.extractall('/kaggle/working/')
    for root, dirs, files in os.walk('/kaggle/working/'):
        if 'models' in dirs and 'checkpoints' in dirs:
            code_dir = root
            break
    if code_dir is None:
        code_dir = ANIMEINTERP_DIR
    print(f'[OK] Giải nén xong: {code_dir}')

else:
    print('[INFO] Clone từ GitHub...')
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/lisiyao21/AnimeInterp.git',
         '/kaggle/working/AnimeInterp'],
        check=True
    )
    code_dir = '/kaggle/working/AnimeInterp'
    print(f'[OK] Clone xong: {code_dir}')

assert code_dir is not None and os.path.isdir(code_dir)

if code_dir not in sys.path:
    sys.path.insert(0, code_dir)
print(f'sys.path[0] = {sys.path[0]}')

In [ ]:
# ============================================================
# CELL 2C: Load AnimeInterp Model
# ============================================================
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[INFO] Sử dụng thiết bị: {DEVICE}')

# Tìm checkpoint
ckpt_path = None
candidates = [
    CHECKPOINT,
    os.path.join(code_dir, 'checkpoints', 'anime_interp_full.ckpt'),
]
for c in candidates:
    if os.path.exists(c):
        ckpt_path = c
        break

if ckpt_path is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if 'anime_interp' in f.lower() and f.endswith(('.ckpt', '.pth')):
                ckpt_path = os.path.join(root, f)
                break
        if ckpt_path:
            break

assert ckpt_path and os.path.exists(ckpt_path), f'Checkpoint not found!'
print(f'[OK] Checkpoint: {ckpt_path} ({os.path.getsize(ckpt_path)/1024**2:.1f} MB)')


def load_animeinterp_model(ckpt_path, device):
    try:
        from models.AnimeInterp_no_cupy import AnimeInterpNoCupy
        model = AnimeInterpNoCupy(path=None)
        print('[INFO] Using AnimeInterpNoCupy (safe for Kaggle)')
    except ImportError:
        from models.AnimeInterp import AnimeInterp
        model = AnimeInterp(path=None)
        print('[INFO] Using AnimeInterp (with cupy)')

    ckpt = torch.load(ckpt_path, map_location=device)
    if isinstance(ckpt, dict):
        state_dict = ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt))
    else:
        state_dict = ckpt

    clean_sd = {k.replace('module.', ''): v for k, v in state_dict.items()}
    missing, unexpected = model.load_state_dict(clean_sd, strict=False)

    if missing:
        print(f'  ⚠️  Missing keys ({len(missing)}): {missing[:3]}')
    model = model.to(device)
    model.eval()

    total_params = sum(p.numel() for p in model.parameters())
    print(f'[OK] Model loaded! Params: {total_params:,} ({total_params/1e6:.1f}M)')
    return model


model = load_animeinterp_model(ckpt_path, DEVICE)
print(f'\n✅ AnimeInterp ready on {DEVICE}!')

---
## 📂 CELL 3: Dataloader Triplet ATD-12K

In [ ]:
# ============================================================
# CELL 3: Dataset & DataLoader cho triplet ATD-12K
# ============================================================
import os
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

# Kích thước target phải chia hết 32
TARGET_W, TARGET_H = 960, 544   # 540 -> 544 (pad để chia hết 32)
assert TARGET_W % 32 == 0 and TARGET_H % 32 == 0

USE_SGM_FLOW = os.path.exists(FLOW_ROOT)
print(f'[CONFIG] Target: {TARGET_W}x{TARGET_H}, SGM Flow: {"Enabled" if USE_SGM_FLOW else "Disabled"}')


class ATD12KTripletDataset(Dataset):
    """
    Dataset đọc triplet (frame1, frame2, frame3) từ cấu trúc ATD-12K.
    frame1=I0 (t=0), frame2=GT (t=0.5), frame3=I1 (t=1)
    """
    def __init__(self, root, flow_root=None, target_w=960, target_h=544):
        self.root      = root
        self.flow_root = flow_root
        self.target_w  = target_w
        self.target_h  = target_h

        all_clips = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
        self.clips = []

        for clip in all_clips:
            clip_dir = os.path.join(root, clip)
            f1 = os.path.join(clip_dir, 'frame1.png')
            f2 = os.path.join(clip_dir, 'frame2.png')
            f3 = os.path.join(clip_dir, 'frame3.png')
            # fallback to .jpg
            if not os.path.exists(f1): f1 = f1.replace('.png', '.jpg')
            if not os.path.exists(f2): f2 = f2.replace('.png', '.jpg')
            if not os.path.exists(f3): f3 = f3.replace('.png', '.jpg')
            if all(os.path.exists(f) for f in [f1, f2, f3]):
                self.clips.append((clip, f1, f2, f3))

        print(f'[Dataset] Valid clips: {len(self.clips)}')
        if len(self.clips) == 0:
            raise RuntimeError(f'No valid clips at: {root}')

    def _load_image(self, path):
        img = Image.open(path).convert('RGB')
        img = img.resize((self.target_w, self.target_h), Image.LANCZOS)
        return TF.to_tensor(img)  # [3, H, W] float [0,1]

    def _load_flow(self, npy_path):
        if not os.path.exists(npy_path):
            return torch.zeros(2, self.target_h, self.target_w)
        flow_np = np.load(npy_path).astype(np.float32)
        h0, w0 = flow_np.shape[1], flow_np.shape[2]
        flow = torch.from_numpy(flow_np).unsqueeze(0)
        flow = F.interpolate(flow, size=(self.target_h, self.target_w), mode='bilinear', align_corners=True)[0]
        flow[0] *= self.target_w / w0
        flow[1] *= self.target_h / h0
        return flow  # [2, H, W]

    def __len__(self): return len(self.clips)

    def __getitem__(self, idx):
        clip_name, f1, f2, f3 = self.clips[idx]
        I0 = self._load_image(f1)
        GT = self._load_image(f2)
        I1 = self._load_image(f3)

        if self.flow_root:
            fd = os.path.join(self.flow_root, clip_name)
            F12i = self._load_flow(os.path.join(fd, 'guide_flo13.npy'))
            F21i = self._load_flow(os.path.join(fd, 'guide_flo31.npy'))
        else:
            F12i = torch.zeros(2, self.target_h, self.target_w)
            F21i = torch.zeros(2, self.target_h, self.target_w)

        return {'I0': I0, 'I1': I1, 'GT': GT, 'F12i': F12i, 'F21i': F21i, 'clip_name': clip_name}


dataset = ATD12KTripletDataset(
    root=DATASET_ROOT,
    flow_root=FLOW_ROOT if USE_SGM_FLOW else None,
    target_w=TARGET_W, target_h=TARGET_H
)

dataloader = DataLoader(
    dataset, batch_size=1, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)
print(f'\n✅ DataLoader ready: {len(dataset)} triplet clips')

# Visualize mẫu
import matplotlib.pyplot as plt
sample = next(iter(dataloader))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key, title in zip(axes, ['I0', 'GT', 'I1'],
                          ['frame1 (I0, t=0)', 'frame2 (GT, t=0.5)', 'frame3 (I1, t=1)']):
    ax.imshow(sample[key][0].permute(1,2,0).numpy())
    ax.set_title(f'{title}\n{sample["clip_name"][0]}', fontsize=10)
    ax.axis('off')
plt.suptitle('Sample Triplet Preview', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_triplet.png'), dpi=100)
plt.show()

---
## 📐 CELL 4: Module Tính Toán 17 Chỉ Số Metrics

In [ ]:
# ============================================================
# CELL 4: Metric functions chuẩn hóa cho AnimeInterp
# ============================================================
import torch
import numpy as np
import lpips
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import time

# Init LPIPS (AlexNet backbone)
print('[INFO] Khởi tạo LPIPS (AlexNet)...')
lpips_fn = lpips.LPIPS(net='alex').to(DEVICE)
lpips_fn.eval()
for p in lpips_fn.parameters():
    p.requires_grad_(False)
print('[OK] LPIPS initialized!')


def tensor_to_numpy_uint8(tensor):
    """Tensor [3,H,W] float[0,1] -> numpy uint8 [H,W,3] RGB"""
    arr = tensor.detach().cpu().float().clamp(0, 1).numpy()
    return (arr * 255).astype(np.uint8).transpose(1, 2, 0)


def compute_psnr(pred_np, gt_np):
    """PSNR (dB). Higher = better."""
    return float(peak_signal_noise_ratio(
        gt_np.astype(np.float64)/255, pred_np.astype(np.float64)/255, data_range=1.0
    ))


def compute_ssim(pred_np, gt_np):
    """SSIM [0,1]. Higher = better."""
    return float(structural_similarity(
        gt_np.astype(np.float64)/255, pred_np.astype(np.float64)/255,
        channel_axis=2, gaussian_weights=True, data_range=1.0
    ))


def compute_lpips(pred_t, gt_t, fn, device):
    """LPIPS perceptual loss. Lower = better."""
    p = (pred_t.unsqueeze(0).to(device) * 2 - 1).clamp(-1, 1)
    g = (gt_t.unsqueeze(0).to(device) * 2 - 1).clamp(-1, 1)
    with torch.no_grad():
        return float(fn(p, g).item())


def compute_ie(pred_np, gt_np):
    """Interpolation Error (ATD-12K formula). Lower = better."""
    p, g = pred_np.astype(np.float64), gt_np.astype(np.float64)
    return float(np.mean(np.sqrt(np.sum((p - g)**2, axis=2))))


@torch.no_grad()
def run_inference_timed(model, I0, I1, F12i, F21i, device, t=0.5, n_warmup=3):
    """
    Inference với đo latency chính xác (loại bỏ GPU warmup).
    Returns: pred[3,H,W], latency_ms, vram_mb
    """
    I0_d   = I0.unsqueeze(0).to(device)
    I1_d   = I1.unsqueeze(0).to(device)
    F12i_d = F12i.unsqueeze(0).to(device)
    F21i_d = F21i.unsqueeze(0).to(device)

    # Warmup
    torch.cuda.synchronize(device)
    for _ in range(n_warmup):
        _ = model(I0_d, I1_d, F12i_d, F21i_d, t)
    torch.cuda.synchronize(device)

    # Reset VRAM counter
    torch.cuda.reset_peak_memory_stats(device)

    # Timed inference
    torch.cuda.synchronize(device)
    t0 = time.perf_counter()
    outputs = model(I0_d, I1_d, F12i_d, F21i_d, t)
    torch.cuda.synchronize(device)
    t1 = time.perf_counter()

    latency_ms = (t1 - t0) * 1000
    vram_mb    = torch.cuda.max_memory_allocated(device) / 1024**2
    pred       = outputs[0].squeeze(0).cpu().float().clamp(0, 1)  # [3,H,W]

    return pred, latency_ms, vram_mb


# Sanity test
print('\n[TEST] Sanity check inference...')
sb = next(iter(dataloader))
pred_t, ms, vram = run_inference_timed(
    model, sb['I0'][0], sb['I1'][0], sb['F12i'][0], sb['F21i'][0], DEVICE, n_warmup=2
)
pnp = tensor_to_numpy_uint8(pred_t)
gnp = tensor_to_numpy_uint8(sb['GT'][0])
print(f'  Output shape : {pred_t.shape}, range [{pred_t.min():.3f}, {pred_t.max():.3f}]')
print(f'  Latency      : {ms:.1f} ms ({1000/ms:.1f} FPS)')
print(f'  VRAM peak    : {vram:.0f} MB')
print(f'  PSNR         : {compute_psnr(pnp, gnp):.2f} dB')
print(f'  SSIM         : {compute_ssim(pnp, gnp):.4f}')
print(f'  LPIPS        : {compute_lpips(pred_t, sb["GT"][0], lpips_fn, DEVICE):.4f}')
print(f'  IE           : {compute_ie(pnp, gnp):.4f}')
print('✅ Sanity test passed!')

---
## 🏃 CELL 5: Pipeline Benchmark Tự Động & Xuất Báo Cáo 17 Chỉ Số

In [ ]:
# ============================================================
# CELL 5A: Vòng lặp benchmark chính
# ============================================================
import pandas as pd
from tqdm.notebook import tqdm
import gc

# Tham số
T_INTERP    = 0.5
N_WARMUP    = 3
MAX_SAMPLES = None  # None = toàn bộ; đặt số nguyên để debug

COLUMNS_17 = [
    'clip_name', 'CD', 'WCD', 'LPIPS', 'EMD', 'RRLD',
    'PSNR', 'SSIM', 'IE', 'FVD', 'Warping_Error',
    'Hausdorff', 'Line_IoU', 'Disconnection_Rate', 'Arbitrary_t',
    'Infer_ms', 'FPS', 'VRAM_MB', 'Finetune_Speed'
]
NUMERIC_COLS = ['LPIPS', 'PSNR', 'SSIM', 'IE', 'Infer_ms', 'FPS', 'VRAM_MB']

total_samples = len(dataset) if MAX_SAMPLES is None else min(MAX_SAMPLES, len(dataset))

results     = []
errors      = []
preview_buf = []
first_done  = False

print('='*65)
print(f'  AnimeInterp Benchmark | {total_samples} clips | {DEVICE}')
print('='*65)

model.eval()
pbar = tqdm(total=total_samples, desc='Benchmarking AnimeInterp')

for idx, batch in enumerate(dataloader):
    if idx >= total_samples:
        break

    clip_name = batch['clip_name'][0]
    I0, I1, GT = batch['I0'][0], batch['I1'][0], batch['GT'][0]
    F12i, F21i  = batch['F12i'][0], batch['F21i'][0]

    try:
        n_w  = N_WARMUP if not first_done else 0
        pred, ms, vram = run_inference_timed(
            model, I0, I1, F12i, F21i, DEVICE, t=T_INTERP, n_warmup=n_w
        )
        first_done = True
        fps = 1000.0 / ms if ms > 0 else 0

        pred_np = tensor_to_numpy_uint8(pred)
        gt_np   = tensor_to_numpy_uint8(GT)

        psnr_v = compute_psnr(pred_np, gt_np)
        ssim_v = compute_ssim(pred_np, gt_np)
        lpip_v = compute_lpips(pred, GT, lpips_fn, DEVICE)
        ie_v   = compute_ie(pred_np, gt_np)

        if len(preview_buf) < 6:
            preview_buf.append({
                'clip': clip_name,
                'I0_np': tensor_to_numpy_uint8(I0),
                'pred_np': pred_np, 'GT_np': gt_np,
                'I1_np': tensor_to_numpy_uint8(I1),
                'psnr': psnr_v, 'ssim': ssim_v, 'lpips': lpip_v,
                'ie': ie_v, 'ms': ms, 'vram': vram
            })

        row = {
            'clip_name': clip_name,
            'CD': 'N/A', 'WCD': 'N/A',
            'LPIPS': round(lpip_v, 6),
            'EMD': 'N/A', 'RRLD': 'N/A',
            'PSNR': round(psnr_v, 4),
            'SSIM': round(ssim_v, 6),
            'IE': round(ie_v, 4),
            'FVD': 'N/A', 'Warping_Error': 'N/A',
            'Hausdorff': 'N/A', 'Line_IoU': 'N/A',
            'Disconnection_Rate': 'N/A', 'Arbitrary_t': 'N/A',
            'Infer_ms': round(ms, 2), 'FPS': round(fps, 2),
            'VRAM_MB': round(vram, 2), 'Finetune_Speed': 'N/A'
        }

    except RuntimeError as e:
        err_t = 'OOM' if 'out of memory' in str(e).lower() else 'ERROR'
        pbar.write(f'  {err_t} [{clip_name}]: {str(e)[:80]}')
        torch.cuda.empty_cache()
        errors.append({'clip': clip_name, 'error': err_t})
        row = {c: err_t for c in COLUMNS_17}
        row['clip_name'] = clip_name

    results.append(row)

    if isinstance(row.get('PSNR'), float):
        pbar.set_postfix({'PSNR': f"{row['PSNR']:.2f}", 'SSIM': f"{row['SSIM']:.4f}", 'ms': f"{row['Infer_ms']:.0f}"})
    pbar.update(1)

    if (idx + 1) % 100 == 0:
        torch.cuda.empty_cache(); gc.collect()

pbar.close()
print(f'\n✅ Xử lý: {len(results)} clips | Lỗi: {len(errors)}')

In [ ]:
# ============================================================
# CELL 5B: Tổng hợp & Xuất CSV
# ============================================================
import pandas as pd

df = pd.DataFrame(results, columns=COLUMNS_17)
df_num = df[NUMERIC_COLS].copy()
for col in NUMERIC_COLS:
    df_num[col] = pd.to_numeric(df[col], errors='coerce')

# Summary rows
mean_row = {'clip_name': '=== MEAN ==='}
std_row  = {'clip_name': '=== STD ==='}
for col in NUMERIC_COLS:
    mean_row[col] = round(df_num[col].mean(), 6)
    std_row[col]  = round(df_num[col].std(), 6)
for col in COLUMNS_17:
    if col not in mean_row: mean_row[col] = 'N/A'
    if col not in std_row:  std_row[col]  = 'N/A'

df_export = pd.concat([df, pd.DataFrame([mean_row, std_row])], ignore_index=True)
df_export.to_csv(CSV_PATH, index=False)
print(f'[OK] CSV saved: {CSV_PATH}')

# Print summary
print('\n' + '='*65)
print('  📊 AnimeInterp Benchmark Summary — 17 Metrics')
print('='*65)

metrics_info = [
    ('01', 'Chamfer Distance (CD)',   '❌', 'N/A'),
    ('02', 'Weighted CD (WCD)',        '❌', 'N/A'),
    ('03', 'LPIPS',                    '✅', f"{df_num['LPIPS'].mean():.4f} ± {df_num['LPIPS'].std():.4f}"),
    ('04', 'EMD',                      '❌', 'N/A'),
    ('05', 'RRLD',                     '❌', 'N/A'),
    ('06', 'PSNR (dB)',                '✅', f"{df_num['PSNR'].mean():.4f} ± {df_num['PSNR'].std():.4f}"),
    ('07', 'SSIM',                     '✅', f"{df_num['SSIM'].mean():.4f} ± {df_num['SSIM'].std():.4f}"),
    ('08', 'IE (Interpolation Error)', '⚠️', f"{df_num['IE'].mean():.4f} ± {df_num['IE'].std():.4f}"),
    ('09', 'FVD',                      '⚠️', 'N/A (single-frame)'),
    ('10', 'Warping Error',            '⚠️', 'N/A (single-frame)'),
    ('11', 'Hausdorff Distance',       '❌', 'N/A'),
    ('12', 'Line IoU',                 '❌', 'N/A'),
    ('13', 'Disconnection Rate',       '❌', 'N/A'),
    ('14', 'Arbitrary t Support',      '❌', 'N/A (fixed t=0.5)'),
    ('15', 'Inference Speed',          '✅', f"{df_num['Infer_ms'].mean():.1f}ms / {df_num['FPS'].mean():.1f}FPS"),
    ('16', 'VRAM Peak (MB)',           '✅', f"{df_num['VRAM_MB'].mean():.0f}"),
    ('17', 'Fine-tuning Speed',        '⚠️', 'N/A'),
]

print(f'  {"#":>3}  {"Chỉ Số":<35} {"Status":^6}  {"Kết Quả"}')
print('  ' + '-'*68)
for num, name, status, val in metrics_info:
    print(f'  {num:>3}  {name:<35} {status:^6}  {val}')

print(f'\n  Clips: {len(df)} | Success: {df_num.dropna(how="all").shape[0]} | Errors: {len(errors)}')
print('='*65)

print('\nPreview (first 5 rows):')
display(df[['clip_name', 'PSNR', 'SSIM', 'LPIPS', 'IE', 'Infer_ms', 'FPS', 'VRAM_MB']].head())

In [ ]:
# ============================================================
# CELL 5C: Visualization — Frame Preview & Metric Distributions
# ============================================================
import matplotlib.pyplot as plt
import matplotlib

plt.style.use('dark_background')
matplotlib.rcParams.update({'figure.facecolor': '#1a1b26', 'axes.facecolor': '#24283b'})


# --- Panel 1: Frame comparison ---
if preview_buf:
    n = min(3, len(preview_buf))
    fig, axes = plt.subplots(n, 4, figsize=(22, 5.5*n), facecolor='#1a1b26')
    if n == 1: axes = [axes]
    fig.suptitle('AnimeInterp — Frame Comparison\n[I0 → Predicted vs GT ← I1]',
                 fontsize=13, fontweight='bold', color='#7dcfff')

    for i, item in enumerate(preview_buf[:n]):
        imgs    = [item['I0_np'], item['pred_np'], item['GT_np'], item['I1_np']]
        labels  = ['Input I0 (t=0)',
                   f'Predicted (t=0.5)\nPSNR={item["psnr"]:.2f}dB SSIM={item["ssim"]:.4f}',
                   f'Ground Truth\nLPIPS={item["lpips"]:.4f} IE={item["ie"]:.2f}',
                   f'Input I1 (t=1)\n{item["ms"]:.0f}ms | {item["vram"]:.0f}MB VRAM']
        colors  = ['#89b4fa', '#f38ba8', '#a6e3a1', '#fab387']
        for j, (img, lbl, col) in enumerate(zip(imgs, labels, colors)):
            axes[i][j].imshow(img)
            axes[i][j].set_title(lbl, color=col, fontsize=9, pad=4)
            axes[i][j].axis('off')

    plt.tight_layout(pad=1.5)
    p1 = os.path.join(OUTPUT_DIR, 'benchmark_preview_frames.png')
    plt.savefig(p1, dpi=120, bbox_inches='tight', facecolor='#1a1b26')
    plt.show()
    print(f'[OK] Frame preview -> {p1}')


# --- Panel 2: Distribution plots ---
plot_cfg = [
    ('PSNR',     'steelblue', 'PSNR (dB) — ↑ Better'),
    ('SSIM',     '#a6e3a1',   'SSIM — ↑ Better'),
    ('LPIPS',    '#f38ba8',   'LPIPS — ↓ Better'),
    ('IE',       '#fab387',   'IE (L2) — ↓ Better'),
    ('Infer_ms', '#89dceb',   'Inference Time (ms)'),
    ('VRAM_MB',  '#cba6f7',   'VRAM Peak (MB)'),
]

fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10), facecolor='#1a1b26')
fig2.suptitle('AnimeInterp — Metric Distributions', fontsize=14, fontweight='bold', color='#7dcfff')

for ax, (col, color, label) in zip(axes2.flatten(), plot_cfg):
    data = df_num[col].dropna()
    if len(data) == 0:
        ax.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=ax.transAxes)
        continue
    ax.hist(data, bins=min(40, max(5, len(data)//3)), color=color, alpha=0.75, edgecolor='#1a1b26')
    ax.axvline(data.mean(), color='white', linestyle='--', linewidth=1.5, label=f'Mean={data.mean():.3f}')
    ax.axvline(data.mean()+data.std(), color=color, linestyle=':', linewidth=1, alpha=0.5)
    ax.axvline(data.mean()-data.std(), color=color, linestyle=':', linewidth=1, alpha=0.5)
    ax.set_title(label, fontweight='bold', fontsize=11)
    ax.legend(fontsize=8)

plt.tight_layout()
p2 = os.path.join(OUTPUT_DIR, 'benchmark_distributions.png')
plt.savefig(p2, dpi=100, bbox_inches='tight', facecolor='#1a1b26')
plt.show()
print(f'[OK] Distributions -> {p2}')


# --- Panel 3: PSNR vs SSIM scatter ---
fig3, ax3 = plt.subplots(figsize=(9, 7), facecolor='#1a1b26')
sc = ax3.scatter(df_num['PSNR'].dropna(), df_num['SSIM'].dropna(),
                  c=df_num['LPIPS'].dropna(), cmap='plasma_r', alpha=0.6, s=25)
plt.colorbar(sc, ax=ax3).set_label('LPIPS (↓ Better)', color='white')
ax3.set_xlabel('PSNR (dB) — ↑ Better'); ax3.set_ylabel('SSIM — ↑ Better')
ax3.set_title('PSNR vs SSIM (color=LPIPS)', fontweight='bold', color='#7dcfff')
ax3.axvline(df_num['PSNR'].mean(), color='white', linestyle='--', alpha=0.3)
ax3.axhline(df_num['SSIM'].mean(), color='white', linestyle='--', alpha=0.3)
plt.tight_layout()
p3 = os.path.join(OUTPUT_DIR, 'benchmark_scatter.png')
plt.savefig(p3, dpi=100, bbox_inches='tight', facecolor='#1a1b26')
plt.show()
print(f'[OK] Scatter -> {p3}')

print('\n✅ BENCHMARK HOÀN TẤT!')
print(f'   CSV: {CSV_PATH}')
print(f'   Plots saved to {OUTPUT_DIR}')

---
## 🛠️ CELL 6: Troubleshooting

In [ ]:
# ============================================================
# CELL 6: Troubleshooting utilities
# ============================================================
print('TROUBLESHOOTING GUIDE — AnimeInterp on Kaggle')
print('='*60)

tips = [
    ('OOM (CUDA Out of Memory)',
     'Giảm TARGET_W/H hoặc dùng gradient checkpoint:\n'
     '  TARGET_W, TARGET_H = 480, 272  # Half resolution\n'
     '  torch.cuda.empty_cache(); gc.collect()'),

    ('ModuleNotFoundError: cupy',
     'Dùng no_cupy version (đã xử lý trong Cell 2C):\n'
     '  from models.AnimeInterp_no_cupy import AnimeInterpNoCupy'),

    ('Size not divisible by 32',
     'Pad tensor trước inference:\n'
     '  ph = (32 - H % 32) % 32; pw = (32 - W % 32) % 32\n'
     '  padded = F.pad(tensor, (0,pw,0,ph), "reflect")'),

    ('PSNR/SSIM = nan',
     'Kiểm tra tensor range:\n'
     '  print(pred.min(), pred.max())  # Phải trong [0,1]\n'
     '  pred = torch.nan_to_num(pred.clamp(0,1))'),

    ('LPIPS > 0.5 (kết quả sai)',
     'Kiểm tra channel order RGB vs BGR:\n'
     '  PIL đọc RGB (đúng) | OpenCV đọc BGR (cần convert)\n'
     '  img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)'),

    ('Checkpoint load thiếu keys',
     'Debug với strict=False:\n'
     '  missing, _ = model.load_state_dict(sd, strict=False)\n'
     '  print("Missing:", missing[:5])'),
]

for i, (title, fix) in enumerate(tips, 1):
    print(f'\n{i}. ❗ {title}')
    print(f'   ✅ Fix: {fix}')

print('\n' + '='*60)
print('Performance benchmarks (typical):')  
print('  T4 GPU  : ~400-600 ms/frame | ~1.5-2.5 FPS')
print('  P100 GPU: ~200-350 ms/frame | ~2.9-5.0 FPS')
print('  VRAM    : ~2000-5000 MB at 960x544')
print('\n  PSNR expected : ~30-33 dB (with SGM flow)')
print('  SSIM expected : ~0.92-0.96')
print('  LPIPS expected: ~0.03-0.08')
print('='*60)
print('\n🚀 Good luck with your benchmark!')

---
## 📚 Tài Liệu Tham Khảo

- **AnimeInterp Paper:** [arxiv.org/abs/2104.02495](https://arxiv.org/abs/2104.02495)
- **GitHub:** [github.com/lisiyao21/AnimeInterp](https://github.com/lisiyao21/AnimeInterp)
- **ATD-12K Dataset:** [Google Drive](https://drive.google.com/file/d/1XBDuiEgdd6c0S4OXLF4QvgSn_XNPwc-g/view)
- **Checkpoint (Dropbox):** [anime_interp_full.ckpt](https://www.dropbox.com/s/oc8juclx1775qib/anime_interp_full.ckpt)

```
AnimeInterp Architecture:
  flownet  : RFR (RAFT-based, iters=12)
  feat_ext : FeatureExtractor (3-scale: 32, 64, 96 channels)
  fwarp    : ForwardWarp (no_cupy version)
  synnet   : GridNet (6ch -> 3ch, 3 resolution levels)
  Ckpt     : anime_interp_full.ckpt (~23.7 MB)
```

---
*PBL6 Benchmark Pipeline — AnimeInterp CVPR 2021*